# 06: Combined Best — Hyperparameter Tuning
**Group 11 · COSE474 Deep Learning · Korea University · Spring 2026**

## Overview

Small grid search over two configs:
- **Config A:** LRCN + Aug + TL
- **Config B:** LRCN + TL only (best so far: 23.26%)

**Grid (4 combinations × 2 configs = 8 runs):**

| Run | Hidden | Dropout | LR |
|---|---|---|---|
| 1 | 64 | 0.4 | 1e-4 |
| 2 | 128 | 0.4 | 1e-4 |
| 3 | 64 | 0.5 | 5e-5 |
| 4 | 128 | 0.3 | 1e-4 |

**Previous results:** CNN 13.57% · LRCN 14.34% · Aug 12.40% · WLASL TL 23.26% · Pose-LRCN 25.00%

# Imports & Config

In [ ]:
import os, json, random, csv, gc
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader

torch.cuda.empty_cache()
gc.collect()

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

NUM_FRAMES     = 32
IMG_SIZE       = 224
BATCH          = 8
FEATURE_DIM    = 512
LSTM_LAYERS    = 1
WARMUP_EPOCHS  = 10
NUM_EPOCHS     = 40
PATIENCE       = 8

NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD  = [0.229, 0.224, 0.225]

GRID = [
    {'hidden': 64,  'dropout': 0.4, 'lr': 1e-4},
    {'hidden': 128, 'dropout': 0.4, 'lr': 1e-4},
    {'hidden': 64,  'dropout': 0.5, 'lr': 5e-5},
    {'hidden': 128, 'dropout': 0.3, 'lr': 1e-4},
]

BASE_DIR      = Path('.')
FRAMES_DIR    = BASE_DIR / 'frames'
CKPT_DIR      = BASE_DIR / 'models' / 'checkpoints'
FIGS_DIR      = BASE_DIR / 'results' / 'figures'
LOGS_DIR      = BASE_DIR / 'results' / 'logs'
for d in [CKPT_DIR, FIGS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CKPT_WLASL_LSTM = str(CKPT_DIR / 'lstm_wlasl_pretrained.pth')
print(f'WLASL LSTM ckpt : {CKPT_WLASL_LSTM}')
print(f'Total runs      : {len(GRID)*2}')

# KSL Word Labels

In [ ]:
KSL_WORDS = {
     1:'hi',          2:'what',         3:'meat',         4:'bi bim rice',
     5:'glad',        6:'hobby',        7:'me',           8:'movie',
     9:'face',       10:'see',         11:'name',        12:'read',
    13:'thank',      14:'equal',       15:'sorry',       16:'eat',
    17:'fine',       18:'do effort',   19:'next',        20:'age',
    21:'again',      22:'how many',    23:'day',         24:'good, nice',
    25:'when',       26:'we',          27:'subway',      28:'be friendly',
    29:'bus',        30:'ride',        31:'cell phone',  32:'where',
    33:'number',     34:'location',    35:'guide',       36:'responsibility',
    37:'who',        38:'arrive',      39:'family',      40:'time',
    41:'friend',     42:'help',        43:'know',        44:'work',
    45:'parents',    46:'10 minutes',  47:'walk',        48:'feel',
    49:'think',      50:'money',       51:'teach',       52:'meet',
    53:'education',  54:'want',        55:'rest',        56:'talk',
    57:'hospital',   58:'go',          59:'house',       60:'school',
    61:'like',       62:'bad',         63:'together',    64:'because',
    65:'no',         66:'now',         67:'different',
}
print(f'KSL words: {len(KSL_WORDS)}')

# Dataset & Splits

In [ ]:
class KSLDataset(Dataset):
    def __init__(self, clip_dirs, transform=None,
                 label_map=None, num_frames=NUM_FRAMES):
        self.clips      = clip_dirs
        self.transform  = transform
        self.label_map  = label_map
        self.num_frames = num_frames

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip_path   = self.clips[idx]
        folder_name = os.path.basename(clip_path)
        class_id    = int(folder_name.split('_')[1])
        label       = self.label_map[class_id] if self.label_map else class_id - 1
        frame_files = sorted(os.listdir(clip_path))
        if len(frame_files) < self.num_frames:
            frame_files += [frame_files[-1]] * (self.num_frames - len(frame_files))
        frame_files = frame_files[:self.num_frames]
        tensors = []
        for fname in frame_files:
            img = Image.open(os.path.join(clip_path, fname)).convert('RGB')
            if self.transform:
                img = self.transform(img)
            tensors.append(img)
        return torch.stack(tensors), label


TRAIN_SIGNERS = {f'{i:02d}' for i in range(16)}
VAL_SIGNERS   = {f'{i:02d}' for i in range(16, 20)}

all_clips = sorted([
    str(FRAMES_DIR / d) for d in os.listdir(FRAMES_DIR)
    if (FRAMES_DIR / d).is_dir()
])

present_class_ids = sorted({int(os.path.basename(c).split('_')[1]) for c in all_clips})
LABEL_MAP         = {cid: i for i, cid in enumerate(present_class_ids)}
KSL_CLS           = len(present_class_ids)

train_clips = [c for c in all_clips if os.path.basename(c).split('_')[0] in TRAIN_SIGNERS]
val_clips   = [c for c in all_clips if os.path.basename(c).split('_')[0] in VAL_SIGNERS]

print(f'KSL classes : {KSL_CLS}')
print(f'Train clips : {len(train_clips)}')
print(f'Val clips   : {len(val_clips)}')

base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

aug_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

val_ds     = KSLDataset(val_clips, base_transform, LABEL_MAP)
val_loader = DataLoader(val_ds, batch_size=BATCH,
                        shuffle=False, num_workers=4, pin_memory=True)
print('Datasets ready')

# Model

In [ ]:
class LRCN(nn.Module):
    def __init__(self, num_classes, feature_dim=FEATURE_DIM,
                 lstm_hidden=64, lstm_layers=LSTM_LAYERS, dropout=0.4):
        super().__init__()
        vgg           = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        self.features = vgg.features
        self.avgpool  = vgg.avgpool
        for p in self.features.parameters():
            p.requires_grad = False
        self.frame_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(input_size=feature_dim, hidden_size=lstm_hidden,
                            num_layers=lstm_layers, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, num_classes),
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        x = self.features(x)
        x = self.avgpool(x)
        x = self.frame_fc(x)
        x = x.view(B, T, -1)
        _, (h, _) = self.lstm(x)
        return self.classifier(h[-1])

print('Model defined')

# Helpers

In [ ]:
def run_epoch(model, loader, optimizer, criterion, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for frames, labels in loader:
            frames, labels = frames.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            outputs = model(frames)
            loss    = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / len(loader), correct / total


def build_and_load(hidden, dropout):
    model = LRCN(num_classes=KSL_CLS, lstm_hidden=hidden,
                 dropout=dropout).to(DEVICE)
    pretrain_state = torch.load(CKPT_WLASL_LSTM, map_location=DEVICE)
    lrcn_state     = model.state_dict()
    for key, val in pretrain_state.items():
        if key.startswith('lstm.') and key in lrcn_state \
           and lrcn_state[key].shape == val.shape:
            lrcn_state[key] = val
    model.load_state_dict(lrcn_state)
    return model


def run_single(hidden, dropout, lr, use_aug, label):
    tag        = f'{label}_h{hidden}_d{int(dropout*10)}_lr{lr}'
    ckpt_path  = str(CKPT_DIR / f'nb06_{tag}.pth')
    t          = aug_transform if use_aug else base_transform
    train_ds   = KSLDataset(train_clips, t, LABEL_MAP)
    train_ldr  = DataLoader(train_ds, batch_size=BATCH,
                             shuffle=True, num_workers=4, pin_memory=True)

    model     = build_and_load(hidden, dropout)
    criterion = nn.CrossEntropyLoss()

    # Warmup
    opt_w = torch.optim.Adam([
        {'params': model.lstm.parameters(),       'lr': lr},
        {'params': model.classifier.parameters(), 'lr': lr * 5},
    ], weight_decay=1e-4)
    for _ in range(WARMUP_EPOCHS):
        run_epoch(model, train_ldr, opt_w, criterion, training=True)

    # Unfreeze VGG 24+
    for i, layer in enumerate(model.features.children()):
        if i >= 24:
            for p in layer.parameters(): p.requires_grad = True

    opt_ft = torch.optim.Adam([
        {'params': model.features.parameters(),   'lr': lr * 0.1},
        {'params': model.frame_fc.parameters(),   'lr': lr},
        {'params': model.lstm.parameters(),       'lr': lr},
        {'params': model.classifier.parameters(), 'lr': lr * 5},
    ], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.StepLR(opt_ft, step_size=10, gamma=0.5)

    best_acc, patience_cnt, log = 0, 0, []
    for epoch in range(1, NUM_EPOCHS - WARMUP_EPOCHS + 1):
        tr_loss, tr_acc = run_epoch(model, train_ldr, opt_ft, criterion, True)
        vl_loss, vl_acc = run_epoch(model, val_loader, opt_ft, criterion, False)
        sched.step()
        log.append([epoch, tr_loss, tr_acc, vl_loss, vl_acc])
        if vl_acc > best_acc:
            best_acc     = vl_acc
            patience_cnt = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_cnt += 1
        if patience_cnt >= PATIENCE: break

    with open(str(LOGS_DIR / f'nb06_{tag}.csv'), 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['epoch','tr_loss','tr_acc','vl_loss','vl_acc'])
        w.writerows(log)

    del model
    torch.cuda.empty_cache(); gc.collect()
    return best_acc, ckpt_path

print('Helpers ready')

# Config A — LRCN + Aug + TL

In [ ]:
print('='*60)
print('  CONFIG A: LRCN + Aug + TL')
print('='*60)

results_a = []
for i, p in enumerate(GRID):
    print(f"\nA{i+1}/4 | hidden={p['hidden']} dropout={p['dropout']} lr={p['lr']}")
    acc, ckpt = run_single(p['hidden'], p['dropout'], p['lr'],
                            use_aug=True, label=f'A{i+1}')
    results_a.append({**p, 'run': f'A{i+1}', 'val_acc': acc, 'ckpt': ckpt})
    print(f'  → {acc:.2%}')

best_a = max(results_a, key=lambda x: x['val_acc'])
print(f"\nBest A: h={best_a['hidden']} d={best_a['dropout']} → {best_a['val_acc']:.2%}")

# Config B — LRCN + TL only

In [ ]:
print('='*60)
print('  CONFIG B: LRCN + TL only (no aug)')
print('='*60)

results_b = []
for i, p in enumerate(GRID):
    print(f"\nB{i+1}/4 | hidden={p['hidden']} dropout={p['dropout']} lr={p['lr']}")
    acc, ckpt = run_single(p['hidden'], p['dropout'], p['lr'],
                            use_aug=False, label=f'B{i+1}')
    results_b.append({**p, 'run': f'B{i+1}', 'val_acc': acc, 'ckpt': ckpt})
    print(f'  → {acc:.2%}')

best_b = max(results_b, key=lambda x: x['val_acc'])
print(f"\nBest B: h={best_b['hidden']} d={best_b['dropout']} → {best_b['val_acc']:.2%}")

# Final Summary

In [ ]:
all_results  = results_a + results_b
overall_best = max(all_results, key=lambda x: x['val_acc'])

print('='*60)
print('  NB06 GRID SEARCH COMPLETE')
print('='*60)
print()
print('Config A (LRCN + Aug + TL):')
for r in sorted(results_a, key=lambda x: -x['val_acc']):
    print(f"  {r['run']} h={r['hidden']} d={r['dropout']} lr={r['lr']} → {r['val_acc']:.2%}")
print()
print('Config B (LRCN + TL only):')
for r in sorted(results_b, key=lambda x: -x['val_acc']):
    print(f"  {r['run']} h={r['hidden']} d={r['dropout']} lr={r['lr']} → {r['val_acc']:.2%}")
print()
print('─'*60)
print('Full ablation table:')
print('  CNN baseline        : 13.57%')
print('  LRCN baseline       : 14.34%')
print('  LRCN + Aug          : 12.40%')
print('  LRCN + WLASL TL     : 23.26%')
print(f'  Best Config A       : {best_a["val_acc"]:.2%}')
print(f'  Best Config B       : {best_b["val_acc"]:.2%}')
print(f'  OVERALL BEST (nb06) : {overall_best["val_acc"]:.2%}')
print('  Pose-LRCN (nb07)    : 25.00%')
print('─'*60)
print(f'  Best config: {overall_best["run"]} '
      f'h={overall_best["hidden"]} d={overall_best["dropout"]}')
print(f'  Checkpoint : {overall_best["ckpt"]}')

with open(str(LOGS_DIR / '06_grid_search_results.csv'), 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['run','hidden','dropout','lr','val_acc','ckpt'])
    w.writeheader()
    for r in all_results:
        w.writerow({k: r[k] for k in ['run','hidden','dropout','lr','val_acc','ckpt']})
print('Results saved to results/logs/06_grid_search_results.csv')